In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

ticker = "UNH"
# Exclude the current New York date so an intraday bar never becomes a close.
end_date = datetime.now(ZoneInfo("America/New_York")).date()
prices = yf.download(ticker, start="2022-01-01", end=end_date.isoformat(), auto_adjust=True)
if prices.empty:
    raise RuntimeError(f"No prices returned for {ticker}. Check the ticker and network connection.")

In [ ]:
close = prices["Close"].squeeze("columns").dropna().sort_index() if isinstance(prices["Close"], pd.DataFrame) else prices["Close"].dropna().sort_index()
close = close.rename(ticker)

In [ ]:
returns = close.pct_change().dropna()

In [ ]:
print(returns.head())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12,8))

close.plot(ax=axes[0], title=f"{ticker} Closing Price")
returns.plot(ax=axes[1], title=f"{ticker} Daily Returns")

plt.tight_layout()
plt.show()

In [ ]:
prices_2023 = prices.loc["2023"]

In [ ]:
prices_2023

In [ ]:
Trend = prices.loc['2022-01-01':'2025-12-31', "Close"]

Trend.plot(figsize=(12,8))

plt.title(f"{ticker} adjusted closing price")
plt.xlabel("date")
plt.ylabel("Adjusted Close Price (USD)")
plt.grid(True)
plt.show()

In [ ]:
month_end_close = close.resample("ME").last()
print(month_end_close.head())

In [ ]:
monthly_mean_close = close.resample("ME").mean()
print(monthly_mean_close.head())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

close.plot(
    ax=ax,
    label="Daily Close",
    alpha=0.5
)
monthly_mean_close.plot(
    ax=ax,
    label="Monthly Mean Close",
    marker="o"
)
ax.set_title(f"{ticker} daily and monthly mean closing prices")
ax.set_xlabel("Date")
ax.set_ylabel("Adjusted Price (USD)")
ax.legend()
ax.grid(True)
plt.show()

## Next trading session forecast
Estimate the next adjusted close from returns and volatility available at the latest included close. The backtest refits the model at each forecast date and reserves the final 20% of labeled observations for evaluation.

In [ ]:
import numpy as np

daily = close.pct_change()
features = pd.DataFrame({
    "daily_return": daily,
    "weekly_return": close.pct_change(5),
    "monthly_return": close.pct_change(20),
    "monthly_volatility": daily.rolling(20).std(),
}).dropna()
next_return = daily.shift(-1).reindex(features.index)
labeled = features.loc[next_return.dropna().index]
if len(labeled) < 100:
    raise ValueError("Need at least 100 labeled sessions after the feature warm-up.")

def forecast_return(history, observed_returns, current):
    # Standardize using training history only, then fit ridge with a fixed penalty.
    center = history.mean()
    scale = history.std(ddof=0).replace(0, 1)
    x = ((history - center) / scale).to_numpy()
    y = observed_returns.to_numpy()
    weights = np.linalg.solve(x.T @ x + 10 * np.eye(x.shape[1]), x.T @ (y - y.mean()))
    return float(y.mean() + ((current - center) / scale).to_numpy() @ weights)

In [ ]:
split = int(0.8 * len(labeled))
records = []
for i in range(split, len(labeled)):
    history = labeled.iloc[:i]
    estimate = forecast_return(history, next_return.loc[history.index], labeled.iloc[i])
    today = close.loc[labeled.index[i]]
    records.append((today * (1 + estimate), today))

target_dates = close.index[close.index.get_indexer(labeled.index[split:]) + 1]
backtest = pd.DataFrame(records, index=target_dates, columns=["Model", "Previous close"])
backtest["Actual"] = close.loc[target_dates]
errors = backtest[["Model", "Previous close"]].sub(backtest["Actual"], axis=0)
print(f"{len(backtest)} forecasts from {target_dates[0]:%Y-%m-%d} through {target_dates[-1]:%Y-%m-%d}")
print(pd.DataFrame({"MAE (USD)": errors.abs().mean(), "RMSE (USD)": errors.pow(2).mean().pow(0.5)}).round(3))
backtest.tail(60).plot(figsize=(12, 5), title=f"{ticker} forecasts by target session", ylabel="Adjusted close (USD)")
plt.show()

In [ ]:
latest = features.iloc[-1]
estimate = forecast_return(labeled, next_return.loc[labeled.index], latest)
last_close = close.loc[latest.name]
print(f"{ticker} adjusted close on {latest.name:%Y-%m-%d}: ${last_close:,.2f}")
print(f"Forecast for the following trading session: ${last_close * (1 + estimate):,.2f} ({estimate:+.2%})")